In [ ]:
import glob
from IPython.display import FileLink, Markdown
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
import seaborn as sns
display(Markdown("Imports ✅"))

In [ ]:
def create_folders(folders: list[str]) -> bool:
    try:
        for folder in folders:
            folderpath: str = os.path.join(os.getcwd(), folder)
            if not os.path.exists(folderpath):
                os.makedirs(folderpath, exist_ok=True)
    except:
        print("Folder {folderpath} could not be created.")
        return False
    else:
        display(Markdown("Folders ✅"))
        return True

In [ ]:
filenameCA: str = 'CASTHPI.csv' # https://fred.stlouisfed.org/series/CASTHPI
filenameCO: str = 'COSTHPI.csv' # https://fred.stlouisfed.org/series/COSTHPI
filenameFL: str = 'FLSTHPI.csv' # https://fred.stlouisfed.org/series/FLSTHPI
filenameHI: str = 'HISTHPI.csv' # https://fred.stlouisfed.org/series/HISTHPI
filenameMI: str = 'MISTHPI.csv' # https://fred.stlouisfed.org/series/MISTHPI
filenameNY: str = 'NYSTHPI.csv' # https://fred.stlouisfed.org/series/NYSTHPI
folders: list[str] = ['data', 'results']
all_dfs: list[pd.DataFrame] = []
create_folders(folders)

for one_filename in glob.glob(os.path.join(folders[0], '??STHPI.csv')):
    one_df: pd.DataFrame = pd.read_csv(one_filename, names=['date', 'price'],header=0,parse_dates=['date'])
    one_df['location'] = one_filename[-11:-9]
    all_dfs.append(one_df)
df: pd.DataFrame = pd.concat(all_dfs).set_index(['date', 'location']).sort_index()
if not df.empty:
    display(Markdown("Data ✅"))
else:
    display(Markdown('Error ❌'))

In [ ]:
# On which date, and in which region, were housing prices historically the highest?
# df.loc[df['price'] == df['price'].max()]
df['price'].idxmax()

In [ ]:
# How long ago was that?
pd.Timestamp('now') - df['price'].idxmax()[0]

In [ ]:
# In which region are the housing prices currently the highest?
# df.index # Index
# df.index[-1] # last one
# df.index[-1][0] # the Timestamp
# df.loc[df.index[-1][0]] # retrieve value
# df.loc[df.index[-1][0], 'price'] # retrieve price column
df.loc[df.index[-1][0], 'price'].idxmax()

In [ ]:
# What are the mean and median prices for each location in our data set?
df.groupby(level='location')['price'].agg(['mean', 'median'])

In [ ]:
# What are the mean prices in New York, across the entire data set?
# df.xs('NY', level='location') # get the location
df.xs('NY', level='location')['price'].mean()

In [ ]:
# Between 2000 and 2020, what were the average prices in New York?
df.xs((slice('2000', '2020'), 'NY'), level=['date', 'location'])['price'].mean()

In [ ]:
# Between 2000 and 2020, what were the average prices in each of New York and California?
from pandas import IndexSlice as idx
# dx['2000':'2020', ['CA','NY']] # range
# df.loc[idx['2000':'2020', ['CA','NY']], :] # all columns
df.loc[idx['2000':'2020', ['CA','NY']], :].groupby('location').mean()

In [ ]:
# Since January 2020, how much have home prices changed in each of the six regions?
# df.loc['2020':, 'price'] # columns
# df.loc['2020':, 'price'].groupby(level='location').pct_change() # location and change
# df.loc['2020':, 'price'].groupby(level='location').pct_change(periods=11) # latest of 12 quarters
df.loc['2020':, 'price'].groupby(level='location').pct_change(periods=11).tail(6).sort_values()